In [1]:
import re
import collections
import pprint

import numpy as np

GLOBAL_RANDOM_SEED = 42

### **Задание 1. Корпус и токенизация.**
- Скачать текст.
- Привести к нижнему регистру, разбить на слова регулярным выражением, знаки препинания либо выбросить, либо сделать отдельными токенами (выбор объяснить).
- Статистика: число слов, размер словаря, двадцать самых частых слов.
- Отложить последние 10% текста для проверки, модель их не видит.

In [2]:
def tokenize_text(text):
    word_pattern = r"[а-яё]+(?:-[а-яё]+)*"
    return re.findall(word_pattern, re.sub("ё", "е", text.lower()))


def load_oblomov_tokens():
    with open("oblomov.txt", "r", encoding="utf8") as file:
        return tokenize_text(file.read())


def run_exercise_1():
    np.random.seed(GLOBAL_RANDOM_SEED)

    tokens = load_oblomov_tokens()

    token_counter = collections.Counter(tokens)
    tokens_length = len(tokens)
    train_size = int(tokens_length * 0.9)
    train_tokens = tokens[:train_size]
    test_tokens = tokens[train_size:]

    print(
        f"Общее количество слов - {tokens_length}",
        f"Размер обучающей выборки - {len(train_tokens)}",
        f"Размер тестовой выборки - {len(test_tokens)}",
        f"Размер словаря - {len(token_counter)}",
        "Наиболее часто встречающиеся слова:",
        sep="\n",
    )
    pprint.pprint(token_counter.most_common(20))


run_exercise_1()

Общее количество слов - 147313
Размер обучающей выборки - 132581
Размер тестовой выборки - 14732
Размер словаря - 25471
Наиболее часто встречающиеся слова:
[('и', 6015),
 ('не', 3896),
 ('в', 3524),
 ('он', 2626),
 ('что', 2407),
 ('на', 2400),
 ('с', 2038),
 ('она', 1634),
 ('а', 1574),
 ('как', 1534),
 ('я', 1443),
 ('все', 1012),
 ('обломов', 909),
 ('это', 903),
 ('его', 903),
 ('у', 879),
 ('ты', 843),
 ('да', 820),
 ('к', 812),
 ('так', 809)]


**Выбор обработки пунктуации.**

Знаки препинания **отброшены**. Токенами считаются только слова найденные 
при помощи регулярного выражения `[а-яё]+(?:-[а-яё]+)*`.

Причины:
1. Пунктуация как отдельные токены увеличила бы размер словаря, но почти
все такие "слова" имели бы частоту 1–2 и не давали бы статистически 
осмысленных биграмм.
2. Запятые и точки в методе `generate` дают текст, забитый знаками, который трудно
читать. Для сравнения температур полезнее "чистый" лексический поток.
3. Для метода`logprob` на отложенной части пунктуация тоже не нужна: она не
различает "связный" и "перемешанный" текст, а только добавляет шум
одинакового порядка в оба варианта.

### **Задание 2. Счетчики и вероятности.**
- Посчитать, сколько раз встретилось каждое слово и каждая пара соседних слов.
- Вероятность следующего слова как доля: $P(w \mid v) = \frac{c(v, w)}{c(v)}$.
- Оформить как класс с методами `fit`, `prob(v, w)`, `generate`, `logprob(text)`.
- Разрешены стандартная библиотека и `numpy`.
- Запрещены `nltk`, `sklearn`, `transformers` и любые готовые языковые модели для основной части.

In [3]:
class BigramModel:
    def __init__(self, alpha=0.0):
        if alpha < 0:
            raise ValueError(f"alpha must be greater or equal 0: {alpha}")

        self.alpha = alpha
        self.unigram_counts = {}
        self.bigram_counts = {}

    def fit(self, tokens):
        self.unigram_counts = {}
        self.bigram_counts = {}

        for token in tokens:
            if token not in self.unigram_counts:
                self.unigram_counts[token] = 1
            else:
                self.unigram_counts[token] += 1

        for index in range(len(tokens) - 1):
            lhs, rhs = tokens[index], tokens[index + 1]
            if lhs not in self.bigram_counts:
                self.bigram_counts[lhs] = {rhs: 1}
            elif rhs not in self.bigram_counts[lhs]:
                self.bigram_counts[lhs][rhs] = 1
            else:
                self.bigram_counts[lhs][rhs] += 1

    def prob(self, v, w):
        # Событие v было, а событие w произойдет!
        vocabulary = len(self.unigram_counts)
        word_count = self.unigram_counts.get(v, 0)
        pair_count = self.bigram_counts.get(v, {}).get(w, 0)

        if vocabulary == 0:
            return 0
        if word_count == 0 and self.alpha == 0:
            return 0

        return (pair_count + self.alpha) / (word_count + self.alpha * vocabulary)

    def generate(self, word, length, temperature=1.0):
        if length < 1:
            raise ValueError(f"length must be greater or equal 1: {length}")
        if temperature <= 0:
            raise ValueError(f"temperature must be greater 0: {temperature}")

        sentence = [word]
        for _ in range(length - 1):
            possible_words = self.bigram_counts.get(word)
            if not possible_words:
                break

            words = list(possible_words)
            counts = np.fromiter(possible_words.values(), dtype=float)
            weights = counts ** (1.0 / temperature)
            weights = weights / weights.sum()

            word = np.random.choice(words, p=weights)
            sentence.append(word)

        return sentence

    def logprob(self, text):
        tokens = tokenize_text(text)
        log_probability = 0.0

        for index in range(len(tokens) - 1):
            probability = self.prob(tokens[index], tokens[index + 1])
            if probability <= 0.0:
                return -np.inf
            log_probability += np.log(probability)

        return log_probability

### **Задание 3. Генерация.**
- Пять текстов по тридцать слов от разных стартовых слов.
- Каждое следующее слово выбирается случайно по вероятностям из строки предыдущего.
- Затем температура: возвести вероятности в степень $1/T$ и перенормировать, показать генерацию при $T = 0.5$, $1$, $2$ и описать, что меняется.

In [4]:
def run_exercise_3():
    np.random.seed(GLOBAL_RANDOM_SEED)

    tokens = load_oblomov_tokens()

    temperatures = [0.5, 1.0, 2.0]
    start_words = ["обломов", "скука", "усталость", "лице", "на"]
    generated_length = 30

    model = BigramModel(alpha=0.0)
    model.fit(tokens)

    for temperature in temperatures:
        print(f"Тексты при температуре T = {temperature}")
        for start in start_words:
            generated = model.generate(start, generated_length, temperature)
            text = " ".join(generated)
            print(f"Текст ({len(generated)} слов) для слова '{start}' - \"{text}\"")


run_exercise_3()

Тексты при температуре T = 0.5
Текст (30 слов) для слова 'обломов' - "обломов не о нем она а я не могла бы и не уйдешь ли бы с глубоким вздохом гостья прогневали господа и не было платить взаимностью принимать участие хозяйки отвечал"
Текст (30 слов) для слова 'скука' - "скука не знаю что ж тебе пришел и не было нельзя было бы было еще несколько минут он не могу за что значит что же вдвоем с ольгой и не"
Текст (30 слов) для слова 'усталость' - "усталость ни на него а если б он в уме два несчастья какие щетки и не мог забрать такую нелепость решил что ж я не то на него в обломовке"
Текст (30 слов) для слова 'лице' - "лице он я не было с каким-то ужасам часто свои дела а он не было и не могу прости и не знал что ж ты не договорила с тобой штольц"
Текст (30 слов) для слова 'на' - "на меня а то что он вздохнул что ж ты не мог только что это не было бы не поворотился у меня не то ей-богу славная хозяйка когда он не"
Тексты при температуре T = 1.0
Текст (30 слов) для слова 'обломов' - "облом

**Что меняется с температурой.** 

Температура управляет "остротой" распределения следующего слова: 
веса $p_i$ возводятся в степень $\frac{1}{T}$ и перенормируются.

- **T = 0.5 (низкая).** Распределение заметно острее исходного: масса
перетекает к самым частым продолжениям. Модель чаще выбирает шаблонные
пары (`"не было"`, `"не мог"`, `"что ж"`, `"и не"`), и текст становится 
менее разнообразным: в выводе много повторов служебных конструкций, а 
содержательные слова проскакивают реже. Грамматически связно, но 
информационно бедно - модель повторяет выученные частоты.
- **T = 1.0.** Сэмплирование ровно из обучающего распределения $P(w \mid v)$.
Видны и связные фрагменты (`"указывая на третий раз"`,
`"пробовала заглянуть в самом начале пути"`), и локальные несостыковки -
типичное поведение биграммной модели без дальней памяти: соседние слова
согласуются, но глобальный смысл не держится.
- **T = 2.0 (высокая).** Распределение "выравнивается": редкие продолжения
получают заметно большую вероятность. Текст лексически разнообразнее
(`"жорж занды"`, `"предстоящим переездом"`, `"азиатской моде"`), но
менее связный: чаще встречаются пары, которые в корпусе были редки или
почти не встречались, и появляются странные сочетания.

### **Задание 4. Правдоподобие текста.**
- Для связного предложения из отложенной части и для тех же слов вперемешку посчитать $\log P = \sum_t \log P(w_t \mid w_{t-1})$.
- Вы столкнетесь с нулями: пары, которой не было в обучающей части, дают $\log 0$.
- Добавить сглаживание Лапласа, $P(w \mid v) = (c(v, w) + \alpha) / (c(v) + \alpha V)$, где $V$ размер словаря, и повторить сравнение.
- Одна ячейка с объяснением, почему без сглаживания сравнение невозможно.

In [5]:
def run_exercise_4():
    np.random.seed(GLOBAL_RANDOM_SEED)

    tokens = load_oblomov_tokens()

    tokens_length = len(tokens)
    train_size = int(tokens_length * 0.9)
    train_tokens = tokens[:train_size]
    test_tokens = tokens[train_size:]

    model = BigramModel(alpha=0.0)
    model.fit(train_tokens)

    shuffles = 4
    test_dataset = [test_tokens] + [
        np.random.permutation(test_tokens) for _ in range(shuffles)
    ]

    print(f"Без сглаживания (alpha = {model.alpha}):")
    for test_row in test_dataset:
        text = " ".join(test_row)
        fragment = " ".join(test_row[:20])
        logprob = model.logprob(text)
        print(f'logP = {logprob}, начало текста = "{fragment}"')

    model.alpha = 1.0
    print(f"Со сглаживанием Лапласа (alpha = {model.alpha}):")

    for test_row in test_dataset:
        text = " ".join(test_row)
        fragment = " ".join(test_row[:20])
        logprob = model.logprob(text)
        print(f'logP = {logprob:.3f}, начало текста = "{fragment}"')


run_exercise_4()

Без сглаживания (alpha = 0.0):
logP = -inf, начало текста = "бог а вот двери говорил захар показывая на образ и на дверь я не к тебе пришел я к куме"
logP = -inf, начало текста = "не вскакивает должен она глупейшею взять или и и вы следующие полноте к путем жизни в барина замялся ольги как"
logP = -inf, начало текста = "от не старческому с мой сапоги прижата свету водку и хлопот ильичу скажу стороне стыдливая прощай кроила понимаешь отсюда подались"
logP = -inf, начало текста = "и андрей задумчивое все маятник силы потеря его ответа злоупотреблении живем по сжала домовитой самолюбивой уехать локтем как просияла осмелится"
logP = -inf, начало текста = "и андрей так такая не раз кротость ничто на разве это петух сердилась испытывать вдруг ей жизни она какая-то заглянула"
Со сглаживанием Лапласа (alpha = 1.0):
logP = -139034.632, начало текста = "бог а вот двери говорил захар показывая на образ и на дверь я не к тебе пришел я к куме"
logP = -144684.759, начало текста = "не вскакивает долж

**Почему без сглаживания сравнение невозможно.** 

При $\alpha = 0$ вероятность биграммы, не встретившейся в обучающей 
части, равна $P(w \mid v) = \frac{0}{c(v)} = 0$, а её логарифм - 
это $-\infty$. Отложенная часть (10% текста, около 14 732 слов) почти 
наверняка содержит хотя бы одну биграмму, которой не было в обучении. 
Перемешанные версии - тем более, ведь случайная перестановка разрушает 
практически все реальные пары.

В результате **и связный, и перемешанный тексты получают $\log{P} = -\infty$**.
Это видно в выводе: все пять строк без сглаживания - `logP = -inf`.
Формально "минус бесконечность равна минус бесконечности", и модель
не различает, где порядок слов естественный, а где - случайный. Более
того, у перемешанного текста отсутствующих биграмм *больше*, но $-\infty$ от
этого не становится "более минус бесконечностью" - сумма логарифмов
упирается в $-\infty$ на первой же нулевой биграмме.

Сглаживание Лапласа заменяет нули на $\frac{\alpha}{(c(v) + \alpha \cdot V)}$,
и все логарифмы становятся конечными. Только после этого сравнение 
приобретает смысл: в выводе видно, что связный текст даёт $\log{P} \approx −139035$,
а перемешанные - $\log{P} \approx −144606\dotsc−144685$. Это и
есть количественная демонстрация того, что модель выучила порядок слов:
она приписывает связному тексту заметно большее правдоподобие, чем его
случайным перестановкам.